In [61]:
import numpy as np
import scipy
import scipy.io as sio
import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd
import os
import time

from glmsingle import GLM_single
from glob import glob
from nilearn.glm.first_level import first_level_from_bids
from nilearn.interfaces.fmriprep import load_confounds

In [49]:
# Rename events based on desired analysis
def update_events(models_events, event_type='stimulus'):
    # stimulus events
    if event_type == 'stimulus':
        for sx, sub_events in enumerate(models_events):
            for mx, run_events in enumerate(sub_events):
                run_events['trial_type'] = run_events['stim_file'].str.replace('.wav', '')
                run_events['trial_type'] = run_events['trial_type'].str.replace('-','_')
                
                # drop non-sound events
                run_events.dropna(subset=['trial_type'], inplace=True)
                
                # drop unused columns
                run_events.drop(['stim_file', 'syllable', 'speaker', 'noise_level', 
                                 'response_time', 'correct_key', 'response_key'],
                                axis=1, inplace=True)

        # create stimulus list from updated events.tsv file
        stim_list = sorted([s for s in run_events['trial_type'].unique() if str(s) != 'nan'])
    
    # trial-specific events
    if event_type == 'trial':
        for sx, sub_events in enumerate(models_events):
            for mx, run_events in enumerate(sub_events):
                name_groups = run_events.groupby('stim_file')['stim_file']
                suffix = name_groups.cumcount() + 1
                #repeats = name_groups.transform('size')
                print(suffix)

                run_events['trial_type'] = run_events['stim_file'].str.replace('.wav', '') + \
                                           '_trial' + suffix.map(str)[:-2]
                                           
                run_events['trial_type'] = run_events['trial_type'].str.replace('-','_')
                run_events['trial_type'] = run_events['trial_type'].str.replace('.0','')

        # create stimulus list from updated events.tsv file
        stim_list = sorted([s for s in run_events['trial_type'].unique() if str(s) != 'nan'])

    #print('stim list: ', stim_list)
    return stim_list, models_events


In [13]:
task_label  = 'badaga'
space_label = 'MNI152NLin2009cAsym'
event_type  = 'stimulus' # 'snr', 'sound'

t_acq = 2
t_r = t_acq # same as t_acq since no silent gap in this acquisition

# correct the fmriprep-given slice reference (middle slice, or 0.5)
slice_time_ref = 0.5 * t_acq / t_r

# define bids and fmriprep directories
project_dir = os.path.join('/bgfs/bchandrasekaran/krs228/data/', 
                           'SSP/')
bidsroot = os.path.join(project_dir, 
                        'data_bids')
fmriprep_dir = os.path.join(bidsroot, 
                            'derivatives', 
                            'fmriprep-23.2.1',
                            )
print('bidsroot: ', bidsroot)
print('fmriprep dir:', fmriprep_dir)

# create output directory
bidsderiv_dir = os.path.join(bidsroot, 
                             'derivatives', 
                             'glmsingle', )
if not os.path.exists(bidsderiv_dir):
    os.makedirs(bidsderiv_dir)
print('bidsderiv dir:', bidsderiv_dir)

bidsroot:  /bgfs/bchandrasekaran/krs228/data/SSP/data_bids
fmriprep dir: /bgfs/bchandrasekaran/krs228/data/SSP/data_bids/derivatives/fmriprep-23.2.1
bidsderiv dir: /bgfs/bchandrasekaran/krs228/data/SSP/data_bids/derivatives/glmsingle


In [14]:
fwhm = 0
subject_list = ['SSP058']


In [38]:
subject_id = subject_list[0]
models, models_run_imgs, \
        raw_models_events, \
        models_confounds = first_level_from_bids(bidsroot, 
                                                 task_label, 
                                                 space_label=space_label,
                                                 sub_labels=[subject_id],
                                                 smoothing_fwhm=fwhm,
                                                 derivatives_folder=fmriprep_dir,
                                                 slice_time_ref=slice_time_ref,
                                                 minimize_memory=False)


/scratch/slurm-6248136/ipykernel_3934702/1684836021.py:4: UserWarning: 'slice_time_ref' provided (0.5) is different from the value found in the BIDS dataset (0.174).
Note this may lead to the wrong model specification.
  models_confounds = first_level_from_bids(bidsroot,


In [50]:
# create updated events dataframes
stim_list, models_events = update_events(raw_models_events, 
                                         event_type=event_type)


In [51]:
models

[FirstLevelModel(memory=Memory(location=None), minimize_memory=False,
                 slice_time_ref=0.5, smoothing_fwhm=0, subject_label='SSP058',
                 t_r=2)]

In [52]:
models_run_imgs

[['/bgfs/bchandrasekaran/krs228/data/SSP/data_bids/derivatives/fmriprep-23.2.1/sub-SSP058/func/sub-SSP058_task-badaga_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz',
  '/bgfs/bchandrasekaran/krs228/data/SSP/data_bids/derivatives/fmriprep-23.2.1/sub-SSP058/func/sub-SSP058_task-badaga_run-02_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz',
  '/bgfs/bchandrasekaran/krs228/data/SSP/data_bids/derivatives/fmriprep-23.2.1/sub-SSP058/func/sub-SSP058_task-badaga_run-03_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz']]

In [53]:
models_events[0][0]

,Unnamed: 0,onset,duration,trial_type
0,0,0.000000,0.3,MA_M1_Q
2,2,2.023639,0.3,DA_F1_8
4,4,3.994792,0.3,DA_F1_8
6,6,8.013839,0.3,MA_M2_Q
8,8,10.009397,0.3,GA_M2_n6
...,...,...,...,...
310,310,347.992024,0.3,BA_F1_n2
312,312,350.001924,0.3,BA_F2_Q
314,314,351.989874,0.3,BA_F1_n6
316,316,353.989027,0.3,GA_M2_n6


### Implement GLMsingle

https://github.com/cvnlab/GLMsingle/blob/main/examples/example3_BIDS.ipynb

In [55]:

# create a directory for saving GLMsingle outputs
outputdir_glmsingle = os.path.join(bidsderiv_dir,'examples','exampleBIDS','GLMsingle')

opt = dict()

# set important fields for completeness (but these would be enabled by default)
opt['wantlibrary'] = 1
opt['wantglmdenoise'] = 1
opt['wantfracridge'] = 1

# for the purpose of this example we will keep the relevant outputs in memory
# and also save them to the disk
opt['wantfileoutputs'] = [1,1,1,1]
opt['wantmemoryoutputs'] = [1,1,1,1]

# running python GLMsingle involves creating a GLM_single object
# and then running the procedure using the .fit() routine
glmsingle_obj = GLM_single(opt)

# visualize all the hyperparameters
print(glmsingle_obj.params)

{'wantlibrary': 1, 'wantglmdenoise': 1, 'wantfracridge': 1, 'wantfileoutputs': [1, 1, 1, 1], 'wantmemoryoutputs': [1, 1, 1, 1], 'numforhrf': 50, 'hrfthresh': 0.5, 'hrffitmask': 1, 'R2thresh': 0, 'hrfmodel': 'optimise', 'n_jobs': 1, 'n_pcs': 10, 'n_boots': 100, 'extra_regressors': False, 'chunklen': 50000, 'wanthdf5': 0, 'wantparametric': 0, 'wantpercentbold': 1, 'wantlss': 0, 'brainthresh': [99.0, 0.1], 'brainR2': [], 'brainexclude': False, 'pcR2cutoff': [], 'pcR2cutoffmask': 1, 'pcstop': 1.05, 'fracs': array([1.  , 0.95, 0.9 , 0.85, 0.8 , 0.75, 0.7 , 0.65, 0.6 , 0.55, 0.5 ,
       0.45, 0.4 , 0.35, 0.3 , 0.25, 0.2 , 0.15, 0.1 , 0.05]), 'wantautoscale': 1, 'seed': 1757517297.2572708, 'suppressoutput': 0, 'lambda': 0, 'firdelay': 30, 'firpct': 99}


In [64]:
data = [nib.load(x).get_fdata() for x in models_run_imgs[0]]
design = models_events[0]
stimdur = 0.3
tr = 0.8

In [65]:
start_time = time.time()


if not os.path.exists(outputdir_glmsingle):

    print(f'running GLMsingle...')
    
    # run GLMsingle
    results_glmsingle = glmsingle_obj.fit(
       design,
       data,
       stimdur,
       tr,
       outputdir=outputdir_glmsingle)
    
    # we assign outputs of GLMsingle to the "results_glmsingle" variable.
    # note that results_glmsingle['typea'] contains GLM estimates from an ONOFF model,
    # where all images are treated as the same condition. these estimates
    # could be potentially used to find cortical areas that respond to
    # visual stimuli. we want to compare beta weights between conditions
    # therefore we are not going to include the ONOFF betas in any analyses of 
    # voxel reliability
    
else:
    print(f'loading existing GLMsingle outputs from directory:\n\t{outputdir_glmsingle}')
    
    # load existing file outputs if they exist
    results_glmsingle = dict()
    results_glmsingle['typea'] = np.load(join(outputdir_glmsingle,'TYPEA_ONOFF.npy'),allow_pickle=True).item()
    results_glmsingle['typeb'] = np.load(join(outputdir_glmsingle,'TYPEB_FITHRF.npy'),allow_pickle=True).item()
    results_glmsingle['typec'] = np.load(join(outputdir_glmsingle,'TYPEC_FITHRF_GLMDENOISE.npy'),allow_pickle=True).item()
    results_glmsingle['typed'] = np.load(join(outputdir_glmsingle,'TYPED_FITHRF_GLMDENOISE_RR.npy'),allow_pickle=True).item()

elapsed_time = time.time() - start_time

print(
    '\telapsed time: ',
    f'{time.strftime("%H:%M:%S", time.gmtime(elapsed_time))}'
)

running GLMsingle...


TypeError: '<' not supported between instances of 'str' and 'int'